In [1]:
import olca_schema as o
import olca_ipc as rest
import pandas as pd
import glob 
import os
import numpy as np
import matplotlib.pyplot as plt
import re
from tqdm import tqdm
import time
import logging
import psutil
from pathlib import Path
import threading
import gc

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

process = psutil.Process(os.getpid())

def get_memory_info():
    """Get current memory usage in MB"""
    mem_info = process.memory_info()
    return mem_info.rss / 1024 / 1024 

CLEAR_CACHE_EVERY_N_CALCS = 1 

class TimeoutException(Exception):
    pass

def call_with_timeout(func, args=(), kwargs=None, timeout=120):
    """Execute a function with a timeout. Raises TimeoutException if timeout exceeded."""
    if kwargs is None:
        kwargs = {}
    
    result = [None]
    exception = [None]
    
    def target():
        try:
            result[0] = func(*args, **kwargs)
        except Exception as e:
            exception[0] = e
    
    thread = threading.Thread(target=target, daemon=True)
    thread.start()
    thread.join(timeout)
    
    if thread.is_alive():
        raise TimeoutException(f"Function call timed out after {timeout} seconds")
    
    if exception[0]:
        raise exception[0]
    
    return result[0]

def clear_openlca_cache():
    """Clear OpenLCA's internal cache and memory"""
    try:
        logging.info("Clearing OpenLCA cache and memory")
        # Force garbage collection
        gc.collect()
        logging.info("Cache cleared")
    except Exception as e:
        logging.warning(f"Could not clear cache: {str(e)}")


In [2]:
def extract_diameter_from_name(name):
    match = re.search(r"(\d+)", name)
    if match:
        return float(match.group(1))
    return None

In [3]:
client = rest.Client("http://localhost:8080/olca/api")
method = client.get(o.ImpactMethod,"22db5d07-5266-41aa-88cd-8472bd7570ab")

components = {
    "Air-to-air heat pump": ["fea798b6-2734-4205-ad97-de15aad54485"],
    "Geothermal heat supply": ["81b48c3d-bfcb-4d70-889b-76f76c082568"],
    "HQ to LQ heat conversion substation": ["03cc4b5d-814a-48a8-afd6-8e33fa0d5aad"],
    "Heat pipe":["e64f995e-785b-4676-937c-9f22176dcc92"],
    "LV electricity distribution main": ["21dda521-558f-4810-9ebd-c995030b9458","e4e1ab1f-1b43-4395-83eb-eb63920bed2b"],
    "LV electricity distribution secondary":["df3f20c7-39e5-4553-aa6d-6631305f909d","ce724b78-fc58-45a0-9860-7042f600493f"],
    "Low-voltage electricity supply":["76c71871-e82f-4a1b-b7fc-b35443bf5696"]
    
}

units = {
    "Air-to-air heat pump": "capacity_kw",
    "Geothermal heat supply": "capacity_kw",
    "HQ to LQ heat conversion substation": "capacity_kw",
    "Heat pipe":"distance_m",
    "LV electricity distribution main": "distance_m",
    "LV electricity distribution secondary":"distance_m",
    "Low-voltage electricity supply":"capacity_kw"
}

results = []
method_units = {c.name: c.ref_unit for c in method.impact_categories}
calculation_count = 0

def save_results_to_csv(results_to_save, output_path, mode='w', header=True):
    """Save results to CSV file (append or overwrite)"""
    if not results_to_save:
        logging.warning("No results to save")
        return
    
    df = pd.DataFrame(results_to_save)
    
    # Create directory if it doesn't exist
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    
    if mode == 'a' and os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
        logging.info(f"Appended {len(results_to_save)} results to {output_path}")
    else:
        df.to_csv(output_path, mode='w', header=True, index=False)
        logging.info(f"Saved {len(results_to_save)} results to {output_path}")

def calculation(name, amount, file, diameter=None, output_path=None):
    """Calculate LCA impacts and save results incrementally"""
    global client, calculation_count
    
    if output_path is None:
        raise ValueError("output_path must be specified")
    
    calculation_count += 1
    mem_before = get_memory_info()
    calc_start = time.time()
    
    prod_id = components.get(name)
    if not prod_id:
        logging.warning(f"No product system found for {name}")
        return

    sum_component = {}
    unit_element = {}
    batch_results = []

    for idx, i in enumerate(prod_id):
        logging.info(f"  Processing product system {idx+1}/{len(prod_id)} for {name}")
        try:
            logging.info(f"    [STEP 1] Requesting product system ID: {i}")
            prod_system = call_with_timeout(lambda: client.get(o.ProductSystem, i), timeout=120)
            logging.info(f"    [STEP 2] ✓ Retrieved product system: {prod_system.name}")
            
            logging.info(f"    [STEP 3] Setting up calculation...")
            setup = o.CalculationSetup(
                target=prod_system,
                impact_method=method,
                amount=amount,
                allocation=o.AllocationType.USE_DEFAULT_ALLOCATION,
            )
            logging.info(f"    [STEP 4] ✓ Setup created")
            
            if diameter is not None:
                param = o.Parameter(name="D_inner", value=diameter)
                setup.parameters = [param]
                logging.info(f"    [STEP 5] ✓ Parameter set: D_inner={diameter}")

            logging.info(f"    [STEP 6] Sending calculation request to OpenLCA...")
            wait_start = time.time()
            result = client.calculate(setup)
            logging.info(f"    [STEP 7] ✓ Calculation request submitted, waiting for results...")
            
            result.wait_until_ready()
            wait_elapsed = time.time() - wait_start
            logging.info(f"    [STEP 8] ✓ OpenLCA processing time: {wait_elapsed:.1f}s")
            
            logging.info(f"    [STEP 9] Retrieving impact results...")
            impacts = result.get_total_impacts()
            logging.info(f"    [STEP 10] ✓ Retrieved {len(impacts)} impact categories")

            for impact in impacts:
                key = impact.impact_category.name
                sum_component[key] = sum_component.get(key, 0) + impact.amount
                unit_element[key] = getattr(impact, "unit", None) or impact.impact_category.ref_unit
            
            # Explicitly clean up large objects
            del result
            del impacts
        
        except TimeoutException as e:
            logging.warning(f"TIMEOUT on product system {i}: {str(e)}")
            logging.warning(f"  Skipping this product system and continuing...")
            continue
        finally:
            # Clean up product system object
            del prod_system
            gc.collect()

    for impact_category, impact_amount in sum_component.items():
        batch_results.append({
            "File": os.path.basename(file),
            "Component": f"{name}_DN{diameter}" if diameter else name,
            "Impact Category": impact_category,
            "Amount": impact_amount,
            "Unit": unit_element.get(impact_category, method_units.get(impact_category, ""))
        })
    
    if batch_results:
        is_first_write = not os.path.exists(output_path)
        save_results_to_csv(batch_results, output_path, 
                          mode='w' if is_first_write else 'a',
                          header=is_first_write)
        results.extend(batch_results)
    
    # Clear batch results and force garbage collection
    batch_results.clear()
    sum_component.clear()
    unit_element.clear()
    gc.collect()
    
    if calculation_count % CLEAR_CACHE_EVERY_N_CALCS == 0:
        clear_openlca_cache()
    
    mem_after = get_memory_info()
    total_elapsed = time.time() - calc_start
    mem_delta = mem_after - mem_before
    
    logging.info(f"Completed {name}")
    logging.info(f"Time: {total_elapsed:.1f}s | Memory: {mem_before:.0f}MB → {mem_after:.0f}MB (Δ {mem_delta:+.1f}MB) | Results saved: {len(batch_results)}")

In [ ]:
electricity_priority = [
    "LV electricity distribution main",
    "LV electricity distribution secondary",
    "Low-voltage electricity supply"
]

logging.info("=" * 80)
logging.info("Starting LCA calculation batch")
logging.info(f"Initial memory usage: {get_memory_info():.0f}MB")
logging.info("=" * 80)

calculation_times = []  
total_components = 0

scenarios_dir = "C:/SUSTAINABILITY/Scenarios"
scenario_folders = sorted([f for f in glob.glob(os.path.join(scenarios_dir, "*")) if os.path.isdir(f)])
logging.info(f"Found {len(scenario_folders)} scenario folders")

for scenario_folder in scenario_folders:
    scenario_name = os.path.basename(scenario_folder)
    csv_file = os.path.join(scenario_folder, "outputs", "bill_of_materials.csv")
    output_filename = f"{scenario_name}_lca_results.csv"
    file_output_csv_path = f"C:/results/{output_filename}"
    
    logging.info(f"\n{'='*80}")
    logging.info(f"Processing scenario folder: {scenario_name}")
    logging.info(f"CSV path: {csv_file}")
    logging.info(f"Output CSV: {file_output_csv_path}")
    logging.info(f"{'='*80}")
    
    if not os.path.exists(csv_file):
        logging.warning(f"CSV file not found for scenario: {scenario_name}")
        # Create a blank results CSV with appropriate header and naming
        blank_df = pd.DataFrame(columns=["File", "Component", "Impact Category", "Amount", "Unit"])
        Path(file_output_csv_path).parent.mkdir(parents=True, exist_ok=True)
        blank_df.to_csv(file_output_csv_path, index=False)
        logging.info(f"Created blank CSV for empty scenario: {scenario_name}")
        continue
    
    df = pd.read_csv(csv_file)
    logging.info(f"CSV read successfully. Rows: {len(df)}")

    names = list(df["name"].unique())
    priority_names = [n for n in names if n in electricity_priority]
    other_names = [n for n in names if n not in electricity_priority]
    ordered_names = priority_names + other_names
    
    logging.info(f"Found {len(names)} unique components: {len(priority_names)} priority, {len(other_names)} others")

    for idx, name in enumerate(tqdm(ordered_names, desc="Calculating components", unit="component"), 1):
        total_components += 1
        start_time = time.time()
        logging.info(f"\n[{idx}/{len(ordered_names)}] Processing component: {name}")

        if "DN" in name:
            unit_defined = units.get("Heat pipe")  
            component_name = "Heat pipe"  
            diameter = extract_diameter_from_name(name)
            logging.info(f"Detected Heat pipe with diameter: {diameter}")
        else:
            unit_defined = units.get(name)
            component_name = name
            diameter = None

        if unit_defined is None:
            logging.warning(f"No unit defined for {name}, skipping")
            continue 

        total_value = df.loc[df["name"] == name, unit_defined].sum()
        logging.info(f"Total value from CSV: {total_value} {unit_defined}")

        # Calculate amount based on component type
        if name == "HQ to LQ heat conversion substation":
            amount = (total_value / 40) * 3.6
        elif name == "Air-to-air heat pump":
            amount = total_value * 3.6 * 50 * 2000
        elif name == "Geothermal heat supply":
            amount = total_value * 3.6 * 50 * 8000
        elif name == "Low-voltage electricity supply":
            amount = (total_value / 1000) * 3.6
        else:
            amount = total_value
        
        logging.info(f"Calculated amount for LCA: {amount}")
        
        try:
            calculation(component_name, amount, csv_file, diameter=diameter, output_path=file_output_csv_path)
        except Exception as e:
            logging.error(f"Failed to calculate {name}: {str(e)}")
            continue
        
        elapsed = time.time() - start_time
        calculation_times.append((name, elapsed))
        
        # Check for performance degradation
        if len(calculation_times) > 1:
            first_time = calculation_times[0][1]
            slowdown_factor = elapsed / first_time
            if slowdown_factor > 1.5:
                logging.warning(f"SLOWDOWN DETECTED: {slowdown_factor:.1f}x slower than first calculation!")
                logging.warning(f"    Current memory: {get_memory_info():.0f}MB")

logging.info("\n" + "=" * 80)
logging.info("All calculations complete!")
logging.info(f"Final memory usage: {get_memory_info():.0f}MB")
logging.info(f"Total components processed: {total_components}")
logging.info(f"Total results in memory: {len(results)}")

# Summary of timing trend
if calculation_times:
    logging.info("\nTiming Summary:")
    logging.info(f"  Fastest: {min(t[1] for t in calculation_times):.1f}s")
    logging.info(f"  Slowest: {max(t[1] for t in calculation_times):.1f}s")
    logging.info(f"  Average: {sum(t[1] for t in calculation_times) / len(calculation_times):.1f}s")
    
logging.info("=" * 80)


2026-01-19 09:24:04,843 - INFO - ================================================================================
2026-01-19 09:24:04,843 - INFO - Starting LCA calculation batch
2026-01-19 09:24:04,843 - INFO - Initial memory usage: 142MB
2026-01-19 09:24:04,843 - INFO - ================================================================================
2026-01-19 09:24:04,843 - INFO - Found 4 scenario folders
2026-01-19 09:24:04,843 - INFO - 
2026-01-19 09:24:04,843 - INFO - Processing scenario folder: holstbuurt_2019_district_heating_stedin_1c4dc326
2026-01-19 09:24:04,843 - INFO - CSV path: C:/SUSTAINABILITY/Scenarios\holstbuurt_2019_district_heating_stedin_1c4dc326\outputs\bill_of_materials.csv
2026-01-19 09:24:04,843 - INFO - Output CSV: C:/results/holstbuurt_2019_district_heating_stedin_1c4dc326_lca_results.csv
2026-01-19 09:24:04,843 - INFO - ================================================================================
2026-01-19 09:24:04,859 - INFO - CSV read successfully. Rows

In [5]:
logging.info("\nLoading all generated results from output files...")
output_dir = "C:/results"
output_files = list(Path(output_dir).glob("*_lca_results.csv"))

if output_files:
    all_results = []
    for output_file in output_files:
        logging.info(f"Loading {output_file.name}")
        df = pd.read_csv(output_file)
        all_results.append(df)
        logging.info(f"  Loaded {len(df)} results")
    
    results_df = pd.concat(all_results, ignore_index=True)
    logging.info(f"Loaded {len(results_df)} total results from {len(output_files)} files")
else:
    logging.warning("No LCA results files found. Using in-memory results instead.")
    results_df = pd.DataFrame(results)
    logging.info(f"Using {len(results_df)} in-memory results")


2026-01-19 10:01:02,032 - INFO - 
Loading all generated results from output files...
2026-01-19 10:01:02,036 - INFO - Loading holstbuurt_2013_district_heating_osm_eca3ed7f_lca_results.csv
2026-01-19 10:01:02,050 - INFO -   Loaded 1848 results
2026-01-19 10:01:02,051 - INFO - Loading holstbuurt_2013_district_heating_stedin_a06c7ae5_lca_results.csv
2026-01-19 10:01:02,059 - INFO -   Loaded 1988 results
2026-01-19 10:01:02,060 - INFO - Loading holstbuurt_2013_full_electrification_osm_d1ed183d_lca_results.csv
2026-01-19 10:01:02,065 - INFO -   Loaded 112 results
2026-01-19 10:01:02,065 - INFO - Loading holstbuurt_2013_full_electrification_stedin_1f026a1e_lca_results.csv
2026-01-19 10:01:02,070 - INFO -   Loaded 112 results
2026-01-19 10:01:02,071 - INFO - Loading holstbuurt_2013_hybrid_osm_0b2a7c98_lca_results.csv
2026-01-19 10:01:02,077 - INFO -   Loaded 3276 results
2026-01-19 10:01:02,078 - INFO - Loading holstbuurt_2013_hybrid_stedin_feb38bb0_lca_results.csv
2026-01-19 10:01:02,082 - I